# Лабораторная работа № 4

## Проектирование и сборка аналитической витрины

**Набор данных:** Seoul Bike Sharing Demand, UCI.

**Выбранный аналитический вопрос:**

В какие часы суток и сезоны наблюдается различный уровень почасового спроса на велосипеды и как этот спрос отличается между праздничными и обычными днями?

**Гранулярность витрины:**

Одна строка таблицы `hourly_rentals` представляет один час наблюдения системы велопроката Seoul Bike в определённую календарную дату.

**Составной бизнес-ключ:** `date + hour`.

Основная мера — `rented_bike_count`, то есть количество арендованных велосипедов за конкретный час.

Для ответа на выбранный вопрос используются срезы по часу суток, сезону, праздничному статусу и календарной дате.


In [1]:
SOURCE_INGESTED_ON = "2026-10-13"

STAGING_PATH = (
    "s3://staging/seoul_bike_sharing_demand/"
    f"source_ingested_on={SOURCE_INGESTED_ON}/"
    "staging.parquet"
)

MART_PATH = (
    "s3://mart/seoul_bike_sharing_demand/"
    f"source_ingested_on={SOURCE_INGESTED_ON}/"
    "hourly_rentals.parquet"
)

print("STAGING:", STAGING_PATH)
print("MART:", MART_PATH)


STAGING: s3://staging/seoul_bike_sharing_demand/source_ingested_on=2026-10-13/staging.parquet
MART: s3://mart/seoul_bike_sharing_demand/source_ingested_on=2026-10-13/hourly_rentals.parquet


In [2]:
import os

import duckdb
from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(".env", usecwd=True)

if not env_path:
    raise FileNotFoundError("Файл .env не найден")

load_dotenv(env_path)

connection = duckdb.connect()

connection.execute("INSTALL httpfs")
connection.execute("LOAD httpfs")

s3_endpoint = os.environ["S3_ENDPOINT"]
s3_access_key = os.environ["S3_ACCESS_KEY"]
s3_secret_key = os.environ["S3_SECRET_KEY"]

connection.execute(f"SET s3_endpoint='{s3_endpoint}'")
connection.execute("SET s3_url_style='path'")
connection.execute("SET s3_use_ssl=false")
connection.execute("SET s3_region='us-east-1'")
connection.execute(f"SET s3_access_key_id='{s3_access_key}'")
connection.execute(f"SET s3_secret_access_key='{s3_secret_key}'")

print("DuckDB подключён к SeaweedFS")


DuckDB подключён к SeaweedFS


In [3]:
connection.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW staging_source AS
    SELECT *
    FROM read_parquet('{STAGING_PATH}')
    """
)

print("staging_source создано")


staging_source создано


In [4]:
connection.sql(
    """
    SELECT COUNT(*) AS staging_rows
    FROM staging_source
    """
).show()


┌──────────────┐
│ staging_rows │
│    int64     │
├──────────────┤
│         8760 │
└──────────────┘



In [5]:
connection.sql(
    """
    SELECT *
    FROM staging_source
    ORDER BY date, hour
    LIMIT 5
    """
).show()


┌────────────┬───────────────────┬───────┬───────────────┬──────────────┬───────────────┬────────────────┬─────────────────────────┬───────────────────────┬─────────────┬─────────────┬─────────┬────────────┬─────────────────┬────────────────────┬───────────────────────────────┬───────────────────────────────────────────┐
│    date    │ rented_bike_count │ hour  │ temperature_c │ humidity_pct │ wind_speed_ms │ visibility_10m │ dew_point_temperature_c │ solar_radiation_mj_m2 │ rainfall_mm │ snowfall_cm │ season  │  holiday   │ functioning_day │ source_ingested_on │ quality_flag_invalid_humidity │ quality_flag_functioning_day_inconsistent │
│    date    │       int64       │ int64 │    double     │    int64     │    double     │     int64      │         double          │        double         │   double    │   double    │ varchar │  varchar   │     boolean     │        date        │            boolean            │                  boolean                  │
├────────────┼─────────────────

In [6]:
connection.sql(
    """
    DESCRIBE staging_source
    """
).show()


┌───────────────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│                column_name                │ column_type │  null   │   key   │ default │  extra  │
│                  varchar                  │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ date                                      │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ rented_bike_count                         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ hour                                      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ temperature_c                             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ humidity_pct                              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ wind_speed_ms                             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │


## 2. Модель аналитической витрины

Для ответа на выбранный вопрос используется одна широкая таблица `hourly_rentals`.

**Гранулярность:** одна строка `hourly_rentals` представляет один час наблюдения системы Seoul Bike в определённую календарную дату.

**Составной ключ:** `date + hour`.

**Основная мера:** `rented_bike_count` — количество арендованных велосипедов за один час.

Единица измерения меры — велосипеды за час.

`rented_bike_count` допускается суммировать по непересекающимся часовым наблюдениям, например по суткам, сезонам или праздничному статусу. При этом `COUNT(*)` обозначает количество часовых наблюдений, а не количество аренд.

Для анализа используются срезы:
- час суток;
- сезон;
- праздничный статус;
- календарная дата;
- месяц;
- день недели.

Дополнительно сохраняются признак функционирования сервиса, дата загрузки исходного staging и quality-флаги из ЛР № 3.

Фильтрация строк при построении витрины не выполняется. Исходная гранулярность и количество строк должны сохраниться.


In [7]:
connection.sql(
    """
    SELECT DISTINCT
        season,
        holiday,
        functioning_day
    FROM staging_source
    ORDER BY season, holiday, functioning_day
    """
).show()


┌─────────┬────────────┬─────────────────┐
│ season  │  holiday   │ functioning_day │
│ varchar │  varchar   │     boolean     │
├─────────┼────────────┼─────────────────┤
│ Autumn  │ Holiday    │ false           │
│ Autumn  │ Holiday    │ true            │
│ Autumn  │ No Holiday │ false           │
│ Autumn  │ No Holiday │ true            │
│ Spring  │ Holiday    │ true            │
│ Spring  │ No Holiday │ false           │
│ Spring  │ No Holiday │ true            │
│ Summer  │ Holiday    │ true            │
│ Summer  │ No Holiday │ true            │
│ Winter  │ Holiday    │ true            │
│ Winter  │ No Holiday │ true            │
├─────────┴────────────┴─────────────────┤
│ 11 rows                      3 columns │
└────────────────────────────────────────┘



## 3. Сборка витрины

Витрина формируется из `staging.parquet` без изменения количества строк.

Для удобства последующего анализа из календарной даты вычисляются:
- год;
- месяц;
- ISO-номер дня недели от 1 до 7.

Исходные quality-флаги из ЛР № 3 переносятся без изменения.


In [8]:
connection.execute(
    """
    CREATE OR REPLACE TEMP VIEW hourly_rentals_mart AS

    SELECT
        date,

        EXTRACT(year FROM date)::INTEGER
            AS year,

        EXTRACT(month FROM date)::INTEGER
            AS month,

        EXTRACT(isodow FROM date)::INTEGER
            AS day_of_week,

        hour,

        season,

        holiday,

        functioning_day,

        rented_bike_count,

        quality_flag_invalid_humidity,

        quality_flag_functioning_day_inconsistent,

        source_ingested_on

    FROM staging_source
    """
)

print("hourly_rentals_mart создано")


hourly_rentals_mart создано


In [9]:
connection.sql(
    """
    SELECT
        (SELECT COUNT(*) FROM staging_source)
            AS staging_rows,

        (SELECT COUNT(*) FROM hourly_rentals_mart)
            AS mart_rows
    """
).show()


┌──────────────┬───────────┐
│ staging_rows │ mart_rows │
│    int64     │   int64   │
├──────────────┼───────────┤
│         8760 │      8760 │
└──────────────┴───────────┘



In [10]:
connection.sql(
    """
    DESCRIBE hourly_rentals_mart
    """
).show()


┌───────────────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│                column_name                │ column_type │  null   │   key   │ default │  extra  │
│                  varchar                  │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ date                                      │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ year                                      │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ month                                     │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ day_of_week                               │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ hour                                      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ season                                    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │


In [11]:
connection.sql(
    """
    SELECT *
    FROM hourly_rentals_mart
    ORDER BY date, hour
    LIMIT 10
    """
).show()


┌────────────┬───────┬───────┬─────────────┬───────┬─────────┬────────────┬─────────────────┬───────────────────┬───────────────────────────────┬───────────────────────────────────────────┬────────────────────┐
│    date    │ year  │ month │ day_of_week │ hour  │ season  │  holiday   │ functioning_day │ rented_bike_count │ quality_flag_invalid_humidity │ quality_flag_functioning_day_inconsistent │ source_ingested_on │
│    date    │ int32 │ int32 │    int32    │ int64 │ varchar │  varchar   │     boolean     │       int64       │            boolean            │                  boolean                  │        date        │
├────────────┼───────┼───────┼─────────────┼───────┼─────────┼────────────┼─────────────────┼───────────────────┼───────────────────────────────┼───────────────────────────────────────────┼────────────────────┤
│ 2017-12-01 │  2017 │    12 │           5 │     0 │ Winter  │ No Holiday │ true            │               254 │ false                         │ false     

In [12]:
connection.sql(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE date IS NULL
        ) AS null_date,

        COUNT(*) FILTER (
            WHERE hour IS NULL
        ) AS null_hour

    FROM hourly_rentals_mart
    """
).show()


┌───────────┬───────────┐
│ null_date │ null_hour │
│   int64   │   int64   │
├───────────┼───────────┤
│         0 │         0 │
└───────────┴───────────┘



In [13]:
connection.sql(
    """
    SELECT COUNT(*) AS duplicate_key_groups
    FROM (
        SELECT
            date,
            hour,
            COUNT(*) AS rows_per_key
        FROM hourly_rentals_mart
        GROUP BY date, hour
        HAVING COUNT(*) > 1
    )
    """
).show()


┌──────────────────────┐
│ duplicate_key_groups │
│        int64         │
├──────────────────────┤
│                    0 │
└──────────────────────┘



In [14]:
connection.sql(
    """
    SELECT
        MIN(rented_bike_count) AS min_rentals,
        MAX(rented_bike_count) AS max_rentals,

        MIN(hour) AS min_hour,
        MAX(hour) AS max_hour,

        MIN(month) AS min_month,
        MAX(month) AS max_month,

        MIN(day_of_week) AS min_day_of_week,
        MAX(day_of_week) AS max_day_of_week

    FROM hourly_rentals_mart
    """
).show()


┌─────────────┬─────────────┬──────────┬──────────┬───────────┬───────────┬─────────────────┬─────────────────┐
│ min_rentals │ max_rentals │ min_hour │ max_hour │ min_month │ max_month │ min_day_of_week │ max_day_of_week │
│    int64    │    int64    │  int64   │  int64   │   int32   │   int32   │      int32      │      int32      │
├─────────────┼─────────────┼──────────┼──────────┼───────────┼───────────┼─────────────────┼─────────────────┤
│           0 │        3556 │        0 │       23 │         1 │        12 │               1 │               7 │
└─────────────┴─────────────┴──────────┴──────────┴───────────┴───────────┴─────────────────┴─────────────────┘



In [15]:
connection.sql(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE quality_flag_invalid_humidity
        ) AS invalid_humidity_flags,

        COUNT(*) FILTER (
            WHERE quality_flag_functioning_day_inconsistent
        ) AS functioning_day_flags

    FROM hourly_rentals_mart
    """
).show()


┌────────────────────────┬───────────────────────┐
│ invalid_humidity_flags │ functioning_day_flags │
│         int64          │         int64         │
├────────────────────────┼───────────────────────┤
│                      0 │                     0 │
└────────────────────────┴───────────────────────┘



### Предварительная проверка витрины

Гранулярность исходного staging сохранена: число строк витрины совпадает с числом строк входного набора.

Составной ключ `date + hour` не содержит пропусков и недопустимых повторов.

Диапазоны `hour`, `month` и `day_of_week` соответствуют определениям вычисленных календарных признаков.

Основная мера `rented_bike_count` сохранила диапазон значений исходного staging.

Quality-флаги из ЛР № 3 перенесены без изменения.


## 4. Сохранение витрины в mart

Подготовленная витрина сохраняется в формате Parquet непосредственно в S3-совместимое хранилище SeaweedFS.

Путь результата:

`s3://mart/seoul_bike_sharing_demand/source_ingested_on=2026-10-13/hourly_rentals.parquet`

Повторное выполнение ноутбука должно перезаписывать этот же объект и не создавать дополнительные копии.


In [16]:
connection.execute(
    f"""
    COPY (
        SELECT *
        FROM hourly_rentals_mart
    )
    TO '{MART_PATH}'
    (FORMAT PARQUET)
    """
)

print("Витрина сохранена:", MART_PATH)


Витрина сохранена: s3://mart/seoul_bike_sharing_demand/source_ingested_on=2026-10-13/hourly_rentals.parquet


## 5. Проверка сохранённой витрины

После материализации витрина повторно читается непосредственно из слоя `mart`.

Проверяются количество строк, схема, составной ключ, основная мера, вычисленные календарные признаки, quality-флаги и пригодность витрины для выбранного аналитического вопроса.


In [17]:
connection.sql(
    f"""
    SELECT COUNT(*) AS mart_rows
    FROM read_parquet('{MART_PATH}')
    """
).show()


┌───────────┐
│ mart_rows │
│   int64   │
├───────────┤
│      8760 │
└───────────┘



In [18]:
connection.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{MART_PATH}')
    """
).show()


┌───────────────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│                column_name                │ column_type │  null   │   key   │ default │  extra  │
│                  varchar                  │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ date                                      │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ year                                      │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ month                                     │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ day_of_week                               │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ hour                                      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ season                                    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │


In [19]:
connection.sql(
    f"""
    SELECT
        COUNT(*) FILTER (
            WHERE date IS NULL
        ) AS null_date,

        COUNT(*) FILTER (
            WHERE hour IS NULL
        ) AS null_hour

    FROM read_parquet('{MART_PATH}')
    """
).show()


┌───────────┬───────────┐
│ null_date │ null_hour │
│   int64   │   int64   │
├───────────┼───────────┤
│         0 │         0 │
└───────────┴───────────┘



In [20]:
connection.sql(
    f"""
    SELECT COUNT(*) AS duplicate_key_groups
    FROM (
        SELECT
            date,
            hour,
            COUNT(*) AS rows_per_key

        FROM read_parquet('{MART_PATH}')

        GROUP BY date, hour
        HAVING COUNT(*) > 1
    )
    """
).show()


┌──────────────────────┐
│ duplicate_key_groups │
│        int64         │
├──────────────────────┤
│                    0 │
└──────────────────────┘



In [21]:
connection.sql(
    f"""
    SELECT
        COUNT(*) FILTER (
            WHERE rented_bike_count IS NULL
        ) AS null_rented_bike_count,

        MIN(rented_bike_count) AS min_rentals,

        MAX(rented_bike_count) AS max_rentals

    FROM read_parquet('{MART_PATH}')
    """
).show()


┌────────────────────────┬─────────────┬─────────────┐
│ null_rented_bike_count │ min_rentals │ max_rentals │
│         int64          │    int64    │    int64    │
├────────────────────────┼─────────────┼─────────────┤
│                      0 │           0 │        3556 │
└────────────────────────┴─────────────┴─────────────┘



In [22]:
connection.sql(
    f"""
    SELECT
        MIN(hour) AS min_hour,
        MAX(hour) AS max_hour,

        MIN(month) AS min_month,
        MAX(month) AS max_month,

        MIN(day_of_week) AS min_day_of_week,
        MAX(day_of_week) AS max_day_of_week

    FROM read_parquet('{MART_PATH}')
    """
).show()


┌──────────┬──────────┬───────────┬───────────┬─────────────────┬─────────────────┐
│ min_hour │ max_hour │ min_month │ max_month │ min_day_of_week │ max_day_of_week │
│  int64   │  int64   │   int32   │   int32   │      int32      │      int32      │
├──────────┼──────────┼───────────┼───────────┼─────────────────┼─────────────────┤
│        0 │       23 │         1 │        12 │               1 │               7 │
└──────────┴──────────┴───────────┴───────────┴─────────────────┴─────────────────┘



In [23]:
connection.sql(
    f"""
    SELECT
        COUNT(*) FILTER (
            WHERE quality_flag_invalid_humidity
        ) AS invalid_humidity_flags,

        COUNT(*) FILTER (
            WHERE quality_flag_functioning_day_inconsistent
        ) AS functioning_day_flags

    FROM read_parquet('{MART_PATH}')
    """
).show()


┌────────────────────────┬───────────────────────┐
│ invalid_humidity_flags │ functioning_day_flags │
│         int64          │         int64         │
├────────────────────────┼───────────────────────┤
│                      0 │                     0 │
└────────────────────────┴───────────────────────┘



In [24]:
connection.sql(
    f"""
    SELECT
        season,
        holiday,
        hour,

        COUNT(*) AS observation_hours,

        ROUND(
            AVG(rented_bike_count),
            2
        ) AS avg_rented_bikes

    FROM read_parquet('{MART_PATH}')

    GROUP BY
        season,
        holiday,
        hour

    ORDER BY
        season,
        holiday,
        hour
    """
).show()


┌─────────┬────────────┬───────┬───────────────────┬──────────────────┐
│ season  │  holiday   │ hour  │ observation_hours │ avg_rented_bikes │
│ varchar │  varchar   │ int64 │       int64       │      double      │
├─────────┼────────────┼───────┼───────────────────┼──────────────────┤
│ Autumn  │ Holiday    │     0 │                 5 │            619.6 │
│ Autumn  │ Holiday    │     1 │                 5 │            472.0 │
│ Autumn  │ Holiday    │     2 │                 5 │            353.6 │
│ Autumn  │ Holiday    │     3 │                 5 │            253.8 │
│ Autumn  │ Holiday    │     4 │                 5 │            156.8 │
│ Autumn  │ Holiday    │     5 │                 5 │             97.8 │
│ Autumn  │ Holiday    │     6 │                 5 │            139.0 │
│ Autumn  │ Holiday    │     7 │                 5 │            233.6 │
│ Autumn  │ Holiday    │     8 │                 5 │            387.4 │
│ Autumn  │ Holiday    │     9 │                 5 │            

### Пригодность витрины для аналитического вопроса

Проверочный запрос успешно группирует данные по сезону, праздничному статусу и часу суток.

Для каждой группы возможно определить количество часовых наблюдений и среднее количество арендованных велосипедов.

Следовательно, витрина содержит необходимые меру и срезы для ответа на выбранный вопрос о различиях почасового спроса по времени суток, сезонам и праздничному статусу.

Полный содержательный анализ показателей в рамках ЛР № 4 не выполняется.


## Вывод

В ходе лабораторной работы на основе подготовленного в ЛР № 3 слоя `staging` построена аналитическая витрина `hourly_rentals`.

Гранулярность исходных данных сохранена: одна строка соответствует одному часу наблюдения Seoul Bike в определённую дату.

Количество строк не изменилось:

`8760 staging = 8760 mart`.

Составной ключ `date + hour` не содержит пропусков и повторов.

Основная мера `rented_bike_count` сохранена без изменения, а для анализа добавлены календарные признаки `year`, `month` и `day_of_week`.

Quality-флаги из ЛР № 3 перенесены в витрину без изменения.

Витрина сохранена в SeaweedFS:

`s3://mart/seoul_bike_sharing_demand/source_ingested_on=2026-10-13/hourly_rentals.parquet`

Проверочный запрос подтвердил возможность анализа почасового спроса в разрезе сезона, праздничного статуса и часа суток.
